# Liquid pair candidate pool: local V2

This stage selects a broad pool of two-asset pairs. A third C asset is not required. The pool may share tickers across pairs; it is not a portfolio and does not open trades. A later strategy can hold up to ten qualifying pairs at a time, with limits on repeated ticker and industry exposure.

The original A/B/C workflow remains available in `data.ipynb` and `analysis.ipynb`. This notebook adds only `data/pair_pool.csv`. No price caches or intermediate CSVs are saved.

Local baseline tag: `abc-baseline-2026-10-22`. Development branch: `research/pair-pool-v2`. GitHub stays at the existing version until explicitly requested.

## Data and screening assumptions

- Yahoo industry labels and market caps are **current snapshots**, recorded with retrieval timestamps. This is not a point-in-time 2025 stock universe and can contain survivorship/selection bias.
- All correlation, OLS, EG, residual ADF, spread statistics and historical liquidity use **2022-2025 Close/Volume only**. No 2026 prices are fetched in this stage. Use Close, not adjusted Close.
- Same-industry pairs and related-industry pairs from `industry_relationships.json` are eligible. Related industries do not establish company-specific supplier/customer relationships.
- Both assets must pass market-cap, minimum-price and trailing 60-observation dollar-volume filters. One share must fit the illustrative per-leg budget. Known configured equivalent share classes are reduced to the more liquid class using formation data.
- Correlation must exceed 0.90. EG uses its 5% critical value; the residual ADF uses the fixed -2.86 threshold. Both tests use lag 1 and `autolag=None`. A positive Close OLS beta and a nonzero residual standard deviation are required.
- The 100,000 USD reference long budget approximates one of ten slots in a 1,000,000 USD long allocation. At formation-end prices, the implied hedge order is checked in **both** directions. Each order must be at most 0.5% of the asset's historical average daily dollar volume. These are screening estimates, not execution shares or a slippage model; actual Open prices and liquidity need to be checked before trading.
- The target is up to 300 accepted pairs. Never weaken thresholds to reach the target. A large searched universe does not make the selected pairs independent or guarantee profitable out-of-sample signals.

In [1]:
from pathlib import Path
import logging
import pandas as pd
from IPython.display import display
from portfolio import CointegrationConfig
from pair_pool import PairPoolConfig, PairPoolBuilder

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

In [2]:
config = PairPoolConfig(
    target_pairs=300,
    max_assets=1000,
    min_market_cap=10_000_000_000,           # Current market cap: USD 10 billion
    min_average_dollar_volume=20_000_000,   # Historical mean Close * Volume: USD 20 million
    liquidity_window=60,                   # Last 60 formation trading observations
    min_price=5.0,
    reference_long_amount=100_000,
    max_order_fraction_adv=0.005,           # At most 0.5% of daily dollar volume
    min_correlation=0.90,
    confidence_level=0.95,
    batch_size=20,
    cointegration=CointegrationConfig(
        train_start="2022-01-01",
        train_end="2025-12-31",
        min_observations=750,
        max_lag=1,
        check_i1=False,
    ),
    relationship_file=Path("industry_relationships.json"),
    output_file=Path("data/pair_pool.csv"),
)
print("Target candidate pairs:", config.target_pairs)
print("Both assets: market cap >=", f"${config.min_market_cap:,.0f}")
print("Both assets: historical daily dollar volume >=", f"${config.min_average_dollar_volume:,.0f}")

Target candidate pairs: 300
Both assets: market cap >= $10,000,000,000
Both assets: historical daily dollar volume >= $20,000,000


In [3]:
builder = PairPoolBuilder(config)
result = builder.build()
pair_pool = result["pairs"]

print("Discovered equities:", len(result["universe"]))
print("Formation-liquid equities:", len(result["eligible_assets"]))
print("Industry/related pairs above correlation threshold:", result["candidate_count"])
print("Pairs tested:", result["tested_count"])
print("Accepted pairs:", len(pair_pool), "/", config.target_pairs)
print("Target reached:", result["target_reached"])
print("Saved:", result["output_file"])
if result["events"]:
    print("Source events:", result["events"])
display(pair_pool.head(20))

INFO: Industry screens 10/145; 67 equity records


INFO: Industry screens 20/145; 177 equity records


INFO: Industry screens 30/145; 225 equity records


INFO: Industry screens 40/145; 289 equity records


INFO: Industry screens 50/145; 343 equity records


...

INFO: Accepted 250/300 pairs after 767 tests


INFO: Accepted 275/300 pairs after 903 tests


INFO: Accepted 300/300 pairs after 1041 tests


Discovered equities: 913
Formation-liquid equities: 817
Industry/related pairs above correlation threshold: 1396
Pairs tested: 1041
Accepted pairs: 300 / 300
Target reached: True
Saved: data\pair_pool.csv
Source events: ['missing_history SPCX: No closing prices', 'missing_history SKHY: No closing prices', 'missing_history HONA: No closing prices', 'missing_history VYLR: No closing prices', 'missing_history SUNB: No closing prices', 'missing_history BSP: No closing prices', 'missing_history CBRS: No closing prices', 'missing_history ARXS: No closing prices', 'missing_history PS: No closing prices', 'missing_history FDXF: No closing prices', 'missing_history RML: No closing prices', 'missing_history MAIR: No closing prices', 'missing_history INIO: No closing prices', 'missing_history WSE: No closing prices', 'missing_history PAYP: No closing prices']


,pair_id,y_ticker,x_ticker,scope,alpha,beta,correlation,price_basis,formation_start,formation_end,...,market_cap_asof_y,average_dollar_volume_y,liquidity_date_y,formation_close_y,industry_x,market_cap_x,market_cap_asof_x,average_dollar_volume_x,liquidity_date_x,formation_close_x
0,BCS__NWG,BCS,NWG,related_industry,-0.674957,1.402089,0.987041,Close,2022-01-03,2025-12-31,...,2026-10-22T19:09:25.162673+00:00,1.279668e+08,2025-12-31,25.450001,Banks—Regional,6.718829e+10,2026-10-22T19:09:25.162673+00:00,3.824983e+07,2025-12-31,17.500000
1,AVGO__FLEX,AVGO,FLEX,related_industry,-35.119368,6.274104,0.987007,Close,2022-01-03,2025-12-31,...,2026-10-22T19:09:25.162673+00:00,1.024298e+10,2025-12-31,346.100006,Electronic Components,2.838195e+10,2026-10-22T19:09:25.162673+00:00,3.252957e+08,2025-12-31,60.419998
2,ORLY__AZO,ORLY,AZO,same_industry,-13.257641,0.028732,0.986521,Close,2022-01-03,2025-12-31,...,2026-10-22T19:09:25.162673+00:00,5.103429e+08,2025-12-31,91.209999,Auto Parts,5.277913e+10,2026-10-22T19:09:25.162673+00:00,5.331948e+08,2025-12-31,3391.500000
3,SPGI__MCO,SPGI,MCO,same_industry,99.750244,0.793661,0.986209,Close,2022-01-03,2025-12-31,...,2026-10-22T19:09:25.162673+00:00,7.928293e+08,2025-12-31,494.408691,Financial Data & Stock Exchanges,8.437118e+10,2026-10-22T19:09:25.162673+00:00,4.052632e+08,2025-12-31,510.850006
4,PWR__EME,PWR,EME,same_industry,68.841382,0.544909,0.985916,Close,2022-01-03,2025-12-31,...,2026-10-22T19:09:25.162673+00:00,4.483804e+08,2025-12-31,422.059998,Engineering & Construction,2.899983e+10,2026-10-22T19:09:25.162673+00:00,2.842023e+08,2025-12-31,611.789978
5,HWM__EMBJ,HWM,EMBJ,same_industry,-1.147602,3.119799,0.985779,Close,2022-01-03,2025-12-31,...,2026-10-22T19:09:25.162673+00:00,4.096016e+08,2025-12-31,205.020004,Aerospace & Defense,1.228891e+10,2026-10-22T19:09:25.162673+00:00,5.731338e+07,2025-12-31,64.370003
6,MAR__IHG,MAR,IHG,same_industry,51.363438,1.820863,0.985564,Close,2022-01-03,2025-12-31,...,2026-10-22T19:09:25.162673+00:00,4.436168e+08,2025-12-31,310.239990,Lodging,2.445280e+10,2026-10-22T19:09:25.162673+00:00,2.141460e+07,2025-12-31,140.830002
7,CLS__CRDO,CLS,CRDO,related_industry,-7.868176,1.871109,0.984738,Close,2022-01-27,2025-12-31,...,2026-10-22T19:09:25.162673+00:00,8.862814e+08,2025-12-31,295.609985,Semiconductors,3.047828e+10,2026-10-22T19:09:25.162673+00:00,9.457845e+08,2025-12-31,143.889999
8,GS__BNY,GS,BNY,related_industry,9.947317,7.091037,0.983792,Close,2022-01-03,2025-12-31,...,2026-10-22T19:09:25.162673+00:00,1.705360e+09,2025-12-31,879.000000,Banks—Diversified,9.799296e+10,2026-10-22T19:09:25.162673+00:00,3.837810e+08,2025-12-31,116.089996
9,MCK__COR,MCK,COR,same_industry,-54.665592,2.590302,0.983661,Close,2022-01-03,2025-12-31,...,2026-10-22T19:09:25.162673+00:00,5.741495e+08,2025-12-31,820.289978,Medical Distribution,6.468564e+10,2026-10-22T19:09:25.162673+00:00,4.853062e+08,2025-12-31,337.750000


## Formation spread and future signals

For each row, `spread = Close_y - alpha - beta * Close_x`, and `z = (spread - spread_mean) / spread_std`. The saved mean/std are formation estimates. This notebook does not choose an entry threshold or trade direction.

A future signal strategy should compute signals with information available at the time, execute at a subsequent Open, allow fewer than ten positions when opportunities are absent, and constrain aggregate ticker/industry exposure. Pool size is separate from active position count. Closing both legs on mean reversion also realizes the winning leg, so its tax outcome differs from the legacy loss-leg-only rotation.

`reference_participation_y/x` are the worst of the two reference directions. Liquidity, market-cap provenance and statistical diagnostics remain in the same pair CSV for inspection.

In [4]:
assert pair_pool["pair_id"].is_unique
assert pair_pool["y_ticker"].ne(pair_pool["x_ticker"]).all()
assert len({frozenset(row) for row in pair_pool[["y_ticker", "x_ticker"]].itertuples(index=False, name=None)}) == len(pair_pool)
assert pair_pool["price_basis"].eq("Close").all()
assert pair_pool["used_lag"].eq(1).all()
assert pair_pool["formation_end"].le("2025-12-31").all()
assert pair_pool["correlation"].gt(config.min_correlation).all()
assert pair_pool["eg_statistic"].lt(pair_pool["eg_critical_value"]).all()
assert pair_pool["residual_adf_statistic"].lt(-2.86).all()
assert pair_pool["beta"].gt(0).all()
for side in ["y", "x"]:
    assert pair_pool[f"market_cap_{side}"].ge(config.min_market_cap).all()
    assert pair_pool[f"average_dollar_volume_{side}"].ge(config.min_average_dollar_volume).all()
    assert pair_pool[f"reference_participation_{side}"].le(config.max_order_fraction_adv).all()
print("Verified: unique two-asset pairs, Close formation data, lag 1, EG/ADF and both-side liquidity filters.")

Verified: unique two-asset pairs, Close formation data, lag 1, EG/ADF and both-side liquidity filters.


In [5]:
display(pd.Series(result["asset_rejections"], name="asset_count").sort_values(ascending=False).to_frame())
display(pd.Series(result["pair_rejections"], name="pair_count").sort_values(ascending=False).to_frame())
display(pair_pool.groupby("scope").size().rename("pairs").to_frame())
print("Distinct assets in the accepted pool:", len(set(pair_pool["y_ticker"]) | set(pair_pool["x_ticker"])))
print("Largest reference order / historical daily dollar volume:",
      f"{pair_pool[['reference_participation_y', 'reference_participation_x']].max().max():.2%}")

,asset_count
insufficient_training_history,36
dollar_volume_below_minimum,32
missing_close_or_volume,15
price_below_minimum,6
whole_share_budget_too_small,1
invalid_formation_volume,1


,pair_count
cointegration_not_significant;residual_adf_not_stationary,474
cointegration_not_significant,264
reference_order_exceeds_adv_limit,3


,pairs
scope,
related_industry,173
same_industry,127


Distinct assets in the accepted pool: 224
Largest reference order / historical daily dollar volume: 0.49%
